# Transformer from Scratch & Benchmark vs. LSTM Lab
Цей зошит містить реалізацію архітектури Transformer з нуля, порівняння з LSTM та аналіз механізму уваги.

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, random_split
import numpy as np
import math
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import unicodedata
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SOS_token = 0
EOS_token = 1
PAD_token = 2
MAX_LENGTH = 10

Using device: cpu


## Частина 1: Підготовка даних
**УВАГА:** Сюди необхідно вставити код завантаження даних з попередньої лабораторної роботи (Bahdanau attention lab).

In [49]:
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {"PAD": PAD_token}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "PAD"}
        self.n_words = 3

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

# 1. Допоміжні функції для очищення тексту
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s.strip()

# 2. Функція зчитування файлу
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")
    # ПЕРЕВІР ШЛЯХ ДО ФАЙЛУ: якщо він в іншій папці, зміни 'data/eng-fra.txt'
    lines = open('data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').read().strip().split('\n')
    
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)
        
    return input_lang, output_lang, pairs

# 3. Функції фільтрації
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
           len(p[1].split(' ')) < MAX_LENGTH and \
           p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [p for p in pairs if filterPair(p)]

# 4. Перетворення речення в індекси
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

# 5. Підготовка даних (твоя функція)
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

# 6. Створення DataLoader (твоя функція)
def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    
    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        
    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

# ==========================================
# ЗАПУСК КОДУ:
# ==========================================
batch_size = 32
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

full_dataset = train_dataloader.dataset
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

batch_size = 32

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

full_dataset = train_dataloader.dataset
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


Reading lines...
Read 135842 sentence pairs
Trimmed to 10599 sentence pairs
Counted words:
fra 4346
eng 2804
Train samples: 9539
Val samples: 1060
Reading lines...
Read 135842 sentence pairs
Trimmed to 10599 sentence pairs
Counted words:
fra 4346
eng 2804


## Частина 2: Побудова Transformer з нуля

In [50]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [51]:
def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = F.softmax(scores, dim=-1)
    if dropout is not None:
        attn_weights = dropout(attn_weights)
    output = torch.matmul(attn_weights, value)
    return output, attn_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        if mask is not None:
            mask = mask.unsqueeze(1)
            
        attn_output, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )
        
        attn_output = self.combine_heads(attn_output)
        output = self.W_o(attn_output)
        return output, attn_weights

In [52]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        cross_output, _ = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout2(cross_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))
        return x

In [53]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=128, num_heads=8, num_layers=3, d_ff=512, max_seq_length=10, dropout=0.1):
        super(Transformer, self).__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length
        
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_token)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_token)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length, dropout)
        
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != PAD_token).unsqueeze(1)
        tgt_mask = (tgt != PAD_token).unsqueeze(1)
        
        seq_len = tgt.size(1)
        nopeak_mask = torch.tril(torch.ones(seq_len, seq_len)).bool().to(tgt.device)
        nopeak_mask = nopeak_mask.unsqueeze(0)
        
        # Виправлена маска без зайвого unsqueeze
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        
        src_embedded = self.dropout(self.positional_encoding(self.src_embedding(src) * math.sqrt(self.d_model)))
        tgt_embedded = self.dropout(self.positional_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model)))
        
        enc_output = src_embedded
        for layer in self.encoder_layers:
            enc_output = layer(enc_output, src_mask)
            
        dec_output = tgt_embedded
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, src_mask, tgt_mask)
            
        output = self.fc_out(dec_output)
        return output

## Частина 3: Базова LSTM модель (Basline)

In [54]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.1):
        super(LSTMEncoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, hidden_size, padding_idx=PAD_token)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class LSTMDecoder(nn.Module):
    def __init__(self, output_size, hidden_size, num_layers=2, dropout=0.1):
        super(LSTMDecoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_size = output_size
        self.embedding = nn.Embedding(output_size, hidden_size, padding_idx=PAD_token)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden, cell):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

class Seq2SeqLSTM(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, hidden_size, num_layers=2, dropout=0.1):
        super(Seq2SeqLSTM, self).__init__()
        self.encoder = LSTMEncoder(src_vocab_size, hidden_size, num_layers, dropout)
        self.decoder = LSTMDecoder(tgt_vocab_size, hidden_size, num_layers, dropout)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.output_size
        
        encoder_outputs, hidden, cell = self.encoder(src)
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(src.device)
        decoder_input = tgt[:, 0].unsqueeze(1)
        
        for t in range(1, tgt_len):
            prediction, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, t, :] = prediction
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            decoder_input = tgt[:, t].unsqueeze(1) if teacher_force else prediction.argmax(1).unsqueeze(1)
            
        return outputs

## Частина 4: Тренування та Benchmarking

In [55]:
def train_epoch(model, dataloader, optimizer, criterion, model_type='transformer'):
    model.train()
    total_loss = 0
    for src, tgt in dataloader:
        optimizer.zero_grad()
        if model_type == 'transformer':
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            output = model(src, tgt_input)
            output = output.reshape(-1, output.size(-1))
            tgt_output = tgt_output.reshape(-1)
        else:
            output = model(src, tgt)
            output = output[:, 1:, :].reshape(-1, output.size(-1))
            tgt_output = tgt[:, 1:].reshape(-1)
            
        loss = criterion(output, tgt_output)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, model_type='transformer'):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in dataloader:
            if model_type == 'transformer':
                tgt_input = tgt[:, :-1]
                tgt_output = tgt[:, 1:]
                output = model(src, tgt_input)
                output = output.reshape(-1, output.size(-1))
                tgt_output = tgt_output.reshape(-1)
            else:
                output = model(src, tgt, teacher_forcing_ratio=0.0)
                output = output[:, 1:, :].reshape(-1, output.size(-1))
                tgt_output = tgt[:, 1:].reshape(-1)
                
            loss = criterion(output, tgt_output)
            total_loss += loss.item()
            
    return total_loss / len(dataloader)

def train_model(model, train_loader, val_loader, num_epochs, learning_rate, model_type='transformer', model_name='Model'):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_token)
    
    train_losses, val_losses, epoch_times = [], [], []
    print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")
    
    for epoch in range(1, num_epochs + 1):
        start_time = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, model_type)
        val_loss = evaluate(model, val_loader, criterion, model_type)
        epoch_time = time.time() - start_time
        
        epoch_times.append(epoch_time)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        if epoch % 5 == 0:
            print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.2f}s")
            
    avg_time = np.mean(epoch_times)
    print(f"\nAverage time per epoch: {avg_time:.2f}s")
    print(f"Final train loss: {train_losses[-1]:.4f}")
    print(f"Final val loss: {val_losses[-1]:.4f}")
    return train_losses, val_losses, epoch_times

In [56]:
num_epochs = 30
learning_rate = 0.0001

transformer = Transformer(
    src_vocab_size=input_lang.n_words,
    tgt_vocab_size=output_lang.n_words,
    d_model=128, num_heads=8, num_layers=3, d_ff=512,
    max_seq_length=MAX_LENGTH, dropout=0.1
).to(device)

lstm_model = Seq2SeqLSTM(
    src_vocab_size=input_lang.n_words,
    tgt_vocab_size=output_lang.n_words,
    hidden_size=128, num_layers=2, dropout=0.1
).to(device)

transformer_losses, transformer_val_losses, transformer_times = train_model(
    transformer, train_loader, val_loader, num_epochs, learning_rate, model_type='transformer', model_name='Transformer'
)

lstm_losses, lstm_val_losses, lstm_times = train_model(
    lstm_model, train_loader, val_loader, num_epochs, learning_rate, model_type='lstm', model_name='LSTM Seq2Seq'
)


Training Transformer
Epoch  5 | Train Loss: 1.7697 | Val Loss: 1.7726 | Time: 34.02s
Epoch 10 | Train Loss: 1.4249 | Val Loss: 1.5259 | Time: 33.05s
Epoch 15 | Train Loss: 1.2162 | Val Loss: 1.3433 | Time: 34.26s
Epoch 20 | Train Loss: 1.0478 | Val Loss: 1.2301 | Time: 35.01s
Epoch 25 | Train Loss: 0.9169 | Val Loss: 1.1319 | Time: 34.91s
Epoch 30 | Train Loss: 0.8038 | Val Loss: 1.0698 | Time: 31.28s

Average time per epoch: 33.52s
Final train loss: 0.8038
Final val loss: 1.0698

Training LSTM Seq2Seq
Epoch  5 | Train Loss: 2.4071 | Val Loss: 2.6170 | Time: 19.50s
Epoch 10 | Train Loss: 2.1520 | Val Loss: 2.5077 | Time: 19.39s
Epoch 15 | Train Loss: 2.0044 | Val Loss: 2.3809 | Time: 18.57s
Epoch 20 | Train Loss: 1.8858 | Val Loss: 2.2745 | Time: 19.26s
Epoch 25 | Train Loss: 1.7842 | Val Loss: 2.2185 | Time: 19.31s
Epoch 30 | Train Loss: 1.7052 | Val Loss: 2.1588 | Time: 19.29s

Average time per epoch: 19.25s
Final train loss: 1.7052
Final val loss: 2.1588
